In [1]:
import pandas as pd

# data from https://data.unesco.org/explore/assets/whc001/view/
world_heritage_sites = pd.read_csv("../data/raw/whc001.csv")
print(world_heritage_sites.head(5))
print(world_heritage_sites.shape) # 1273 rows 54 fields
print(world_heritage_sites.dtypes)

print(world_heritage_sites.columns.tolist())
# location data could be redundant ISO Code , Region , Region Code , Transboundary
# a lot of data fields that are translations


                                             Name EN  \
0  The Bony Fish Fossils of the Western Limfjord ...   
1                                        Aalto Works   
2      Beaches of the D-Day Landings, Normandy, 1944   
3                   Ancient Buddhist Site of Sarnath   
4           Alamūt Castle and Related Fortifications   

                                             Name FR  \
0  Les fossiles de poissons osseux du Limfjord oc...   
1                                 Les œuvres d’Aalto   
2        Les Plages du Débarquement, Normandie, 1944   
3                  Site bouddhiste ancien de Sarnath   
4       Château d’Alamūt et fortifications associées   

                                             Name ES  \
0  Los fósiles de peces óseos de Limfjord occiden...   
1                                 Las obras de Aalto   
2         Las Playas del Desembarco, Normandía, 1944   
3                   Antiguo sitio budista de Sarnath   
4  Castillo de Alamūt y fortificaciones relaci

In [2]:
# no duplicate ids
print(world_heritage_sites[world_heritage_sites.duplicated(subset=["ID"], keep=False)])

# looking into missing data

#  show columns with missing values
missing_data = pd.DataFrame({
    "Missing Count": world_heritage_sites.isna().sum(),
    "Missing Percentage": world_heritage_sites.isna().mean() * 100
})

missing_data = missing_data.sort_values(
    by="Missing Percentage",
    ascending=False
)

print(missing_data)
# looking into data with more than 5% missing (apart from image and video data since it's going to get dropped in data preprocessing)
# data end is missing for each column shouldn't be kept

 # danger list has  95.443833% miss rate

endangered_sites = world_heritage_sites[
    world_heritage_sites["Danger list"].notna()
]
 # if danger list isn't missing Danger =  True, with both columns being tightly intertwined, think about feature engineering
print(endangered_sites[["Name EN", "Danger","Danger list" ,"ISO Codes"]].to_string(index=False))
print(endangered_sites["ISO Codes"].value_counts()),
# manually checked if data seems accurate and  countries with more than 1 endangered site have level 3 or 4 threat level https://travel.state.gov/en/international-travel/travel-advisories.html

# natural criteria  77.847604 % missing , cultural criteria 20.345640%  are these missing values related
print(pd.crosstab(
    world_heritage_sites["Cultural Criteria"].notna(),
    world_heritage_sites["Natural Criteria"].notna(),
    margins=True
))
#40 sites have both criteria  ,  17 have none since the 2 categories are so related they can be joined




Empty DataFrame
Columns: [Name EN, Name FR, Name ES, Name RU, Name AR, Name ZH, Short Description EN, Short Description FR, Short Description ES, Short Description RU, Short Description AR, Short Description ZH, Description EN, Justification EN, Date inscribed, Secondary dates, Danger, Date end, Danger list, Area hectares, Cultural Criteria, Natural Criteria, Criteria, Category, Category ID, States Names, ISO Codes, Region, Region Code, Transboundary, Main Image, Main Image Author, Main Image Copyright, Main Image Caption EN, Main Image Caption FR, Main Image Caption ES, Main Image Caption RU, Main Image Caption AR, Main Image Caption ZH, Images, UUID, ID, Coordinates, Main Video, Main Video Author, Main Video Caption EN, Main Video Caption FR, Main Video Caption ES, Main Video Caption RU, Main Video Caption AR, Main Video Caption ZH, videos_urls, Components, Components Count]
Index: []

[0 rows x 54 columns]
                       Missing Count  Missing Percentage
Date end            

In [3]:
# wikipedia data
articles = pd.read_csv("../data/raw/wikipedia_articles.csv")
views = pd.read_csv("../data/raw/wikipedia_views.csv")

print(views.columns)

print(articles.head(5))
print(views.head(5))

# checking types
print(views.dtypes)
print(articles.dtypes)

#  show columns with missing values
missing_views = pd.DataFrame({
    "Missing Count": views.isna().sum(),
    "Missing Percentage": views.isna().mean() * 100
})

print(missing_views)


#from manually clicking it seems a decent portion  of the articles behind the URL's don't exist

# 2. Pageview distribution

print("Pageview rows:", len(views))
print("Unique UNESCO IDs:", views["ID"].nunique())
print("Languages:", views["language"].nunique())

print("\nPageview summary:")
print(views["views"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))

print("\nZero-view records:", (views["views"] == 0).sum())
print("Missing views:", views["views"].isna().sum())

jan_views = views.loc[
    views["month"].astype(str).str[:6] == "202601"
].copy()

site_views = (
    jan_views.groupby("ID", as_index=False)["views"]
    .sum()
    .sort_values("views", ascending=False)
)

print(site_views.head(20))

# Number of distinct language editions represented per site
languages_per_site = (
    jan_views.groupby("ID")["language"]
    .nunique()
    .rename("language_count")
)

print("\nLanguage coverage per site:")
print(languages_per_site.describe())

print("\nSites with the most language editions:")
print(languages_per_site.sort_values(ascending=False).head(20))


Index(['ID', 'language', 'article_title', 'article_url', 'month', 'views'], dtype='str')
     ID wikidata_id language language_name                  article_title  \
0   692     Q263883       en       English    Odzala-Kokoua National Park   
1   692     Q263883       fr        French  Parc national d'Odzala-Kokoua   
2   692     Q263883       es       Spanish         Parque nacional Odzala   
3  1595     Q685357       ru       Russian                     Порт-Ройал   
4   111     Q804881       fr        French   Parc national des monts Balé   

                                         article_url  
0  https://en.wikipedia.org/wiki/Odzala-Kokoua_Na...  
1  https://fr.wikipedia.org/wiki/Parc_national_d%...  
2  https://es.wikipedia.org/wiki/Parque_nacional_...  
3  https://ru.wikipedia.org/wiki/%D0%9F%D0%BE%D1%...  
4  https://fr.wikipedia.org/wiki/Parc_national_de...  
    ID language                article_title  \
0  692       en  Odzala-Kokoua National Park   
1  692       en  Odzal